In [1]:
# todo

# buttons with names of gpx files matched to colour of line to hide/show
# hide all and show all buttons

# hide bokeh pannel on right

In [6]:
import gpxpy
import pandas as pd
from pyproj import Transformer
from bokeh.plotting import figure, show, output_file
from bokeh.models import CustomJS, MultiChoice, Div, Button
from bokeh.layouts import column, row
from pathlib import Path

#from bokeh.tile_providers import get_provider, Vendors
import matplotlib.colors as mcolors
import itertools
mcolors.TABLEAU_COLORS
mcolors.XKCD_COLORS
mcolors.CSS4_COLORS
#Base colors are in RGB so they need to be converted to HEX
BASE_COLORS_hex = {name:mcolors.rgb2hex(color) for name,color in mcolors.BASE_COLORS.items()}
colors = {}
colors.update(mcolors.TABLEAU_COLORS)
#colors.update(mcolors.XKCD_COLORS)

print(colors.values())
colors=itertools.cycle(colors.values())


dict_values(['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf'])


In [ ]:
mcolors.

In [3]:
def read_gpx_file(input_file, max_distance=25):

    gpx_file = open(input_file, 'r')
    gpx = gpxpy.parse(gpx_file)
    gpx.simplify(max_distance=max_distance)
    
    points = []
    for track in gpx.tracks:
        for segment in track.segments:
            for point in segment.points:
                points.append([point.longitude, point.latitude])
    
    df = pd.DataFrame(points, columns=['lon', 'lat'])
    # 2. Convert to Web Mercator (EPSG:3857)
    transformer = Transformer.from_crs("EPSG:4326", "EPSG:3857", always_xy=True)
    df['x'], df['y'] = transformer.transform(df['lon'].values, df['lat'].values)
    return df 

In [4]:
# parse gpxs into dfs
import glob as glob 

gpx_files = sorted(glob.glob('./gpxs/*.gpx'))
#gpx_files = gpx_files[:10]
df_list = []
info_df=pd.read_csv('./gpxs/info.csv')
for f, line in zip(gpx_files,info_df.iterrows()) :
    df= read_gpx_file(f, max_distance=10)
    df.attrs={'index':Path(f).stem, 'i':line[1]['i'], 'name':line[1]['name'] }
    df_list.append(df)





In [9]:
# create plot
p = figure(
           x_axis_type="mercator", y_axis_type="mercator", match_aspect=True)
tile_provider = 'OpenStreetMap Mapnik'
#tile_provider= "CartoDB Positron retina"
p.add_tile(tile_provider, retina=True)
p.ygrid.visible = False
p.xgrid.visible = False
p.axis.visible = False

lines=[]
for df in df_list:
    c = next(colors)
    line=p.line(x='x', y='y', source=df, line_color=c, line_width=4, name= df.attrs['name'])
    lines.append(line)


output_file("gpx_track.html")
#show(p)


# create checkbuttons

 # css stlye for selection buttons
i=1
sheet_text=""
check_labels=[]
for l in lines:
    col = l.glyph.line_color
    #sheet_text=sheet_text+""".bk-btn.bk-active:nth-child(%s){\n background-color: %s;}"""%(i, col) # active button
    sheet_text=sheet_text+""".choices__inner .choices__item.solid:nth-child(%s){\n background-color: %s;}"""%(i, col) # active button
    #sheet_text=sheet_text+""".bk-btn:nth-child(%s){\n color: %s;}"""%(i, colors[i]) # general
    check_labels.append(l.name)
    i=i+1

checkgroup = CheckboxButtonGroup(
    labels=check_labels,
    active=list(range(0,len(lines))),
    stylesheets=[sheet_text],
    orientation='vertical', 
)


multi_choice = MultiChoice(value=[], options=check_labels, stylesheets=[sheet_text], width_policy='fit')


multi_choice.js_on_change( "value", CustomJS(args=dict(lines=lines) ,code="""  
console.log(' changed selected option', cb_obj.active);
    const active_buttons = cb_obj.value;
    for (var i =0 ; i < lines.length; i++){
    
    if (active_buttons.includes(lines[i].name)){
         lines[i].visible=true;
    }
    else{
        lines[i].visible=false;
    }
    }
"""))

checkgroup.js_on_change("active", CustomJS(args=dict(lines=lines,btn=checkgroup), code="""
    console.log('checkbox_button_group: active=' + btn.active, this.toString());
    const active_buttons = btn.active;
    for (var i =0 ; i < lines.length; i++){
    if (active_buttons.includes(i)){
         lines[i].visible=true;
    }
    else{
        lines[i].visible=false;
    }
    }
"""))




# buttons to hide and show all 
button_show = Button(label="Show all", button_type="primary")
button_show.js_on_event("button_click", CustomJS(args=dict(lines=lines, checkbutton_group=checkgroup),
    code=""" 
    checkbutton_group.active=Array.from({length: lines.length}, (x, i) => i);
    for (var i =0 ; i < lines.length; i++){lines[i].visible=true;}""" ))

button_hide = Button(label="Hide all", button_type="primary")
button_hide.js_on_event("button_click", CustomJS(args=dict(lines=lines, checkbutton_group=checkgroup),
    code=""" checkbutton_group.active = [];
    for (var i =0 ; i < lines.length; i++){lines[i].visible=false; }""" ))
layout =  row(p,multi_choice,  button_hide, button_show)
show(layout)

#output_file("gpx_track.html")
#save(layout)

In [8]:
help(MultiChoice)

Help on class MultiChoice in module bokeh.models.widgets.inputs:

class MultiChoice(InputWidget)
 |  MultiChoice widget.
 |  
 |  Method resolution order:
 |      MultiChoice
 |      InputWidget
 |      bokeh.models.widgets.widget.Widget
 |      bokeh.models.layouts.LayoutDOM
 |      bokeh.models.ui.panes.Pane
 |      bokeh.models.ui.ui_element.UIElement
 |      bokeh.models.ui.ui_element.StyledElement
 |      bokeh.model.model.Model
 |      bokeh.core.has_props.HasProps
 |      bokeh.core.serialization.Serializable
 |      bokeh.model.util.HasDocumentRef
 |      bokeh.util.callback_manager.PropertyCallbackManager
 |      bokeh.util.callback_manager.EventCallbackManager
 |      builtins.object
 |  
 |  Methods defined here:
 |  
 |  __init__(*, align='auto', aspect_ratio=None, context_menu=None, css_classes=[], css_variables={}, delete_button=True, description=None, disabled=False, elements=[], flow_mode='block', height=None, height_policy='auto', html_attributes={}, html_id=None, marg

# Testing # 

In [7]:
from bokeh.io import show
from bokeh.models import CustomJS, MultiChoice

OPTIONS = ["foo", "bar", "baz", "quux"]

multi_choice = MultiChoice(value=["foo", "baz"], options=OPTIONS)
multi_choice.js_on_change("value", CustomJS(code="""
    console.log('multi_choice: value=' + this.value, this.toString())
"""))

show(multi_choice)

In [ ]:
bokeh.__version__

In [ ]:
import bokeh

In [ ]:
from bokeh.models import TileSource, WMTSTileSource
import xyzservices

In [ ]:
selected_provider = xyzservices.providers.query_name("openstreetmap_mapnik")

scale_factor=2
tile_source = WMTSTileSource(
                url=selected_provider.build_url(scale_factor=scale_factor),
                attribution=selected_provider.html_attribution,
                min_zoom=selected_provider.get("min_zoom", 0),
                max_zoom=selected_provider.get("max_zoom", 30),
                extra_url_vars={"Referrer-Policy":"strict-origin-when-cross-origin"}
            )

In [ ]:
p = figure(x_range=x_range, y_range=y_range,
           x_axis_type="mercator", y_axis_type="mercator")
p.add_tile(tile_source,  )
show(p)

In [ ]:
selected_provider.items()

In [ ]:
show(p)

In [ ]:
# 1. Parse GPX file
gpx_file = open('./gpxs/001.gpx', 'r')
gpx = gpxpy.parse(gpx_file)
gpx.simplify(max_distance=50)
points = []
for track in gpx.tracks:
    for segment in track.segments:
        for point in segment.points:
            points.append([point.longitude, point.latitude])

df = pd.DataFrame(points, columns=['lon', 'lat'])


print(len(df)),gpx.get_track_points_no()

In [ ]:
gpx.simplify(max_distance=50)

In [ ]:
gpx.get_track_points_no()

In [ ]:
gpx.reduce_points(min_distance=50)

print(len(gpx.get_points_data()),gpx.get_track_points_no() )